In [2]:
!pip install PyPortfolioOpt
!pip install scikit-learn
!pip install stable-baselines3
!pip install gymnasium
!pip install pypfopt
!pip install tqdm
!pip install torch
!pip install torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.1/220.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.1/958.1 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from sklearn.preprocessing import StandardScaler
import torch
from datetime import datetime
from tqdm import tqdm

class StockPortfolioIterator:
    def __init__(self, file_path: str, lookback_window: int = 30, min_history: int = 100, max_stocks: int = 1568, train_years: int = 15):

        print(f"Loading data from {file_path}...")
        self.data = pd.read_csv(file_path)
        self.data['date'] = pd.to_datetime(self.data['date'])

        # Only use recent data (last few years)
        cutoff_date = self.data['date'].max() - pd.DateOffset(years=train_years)
        self.data = self.data[self.data['date'] >= cutoff_date]

        self.data = self.data.sort_values(['date', 'tic'])

        # Filter stocks with enough history
        stock_counts = self.data.groupby('tic').size()
        valid_stocks = stock_counts[stock_counts >= min_history].index

        # Select stocks with highest trading volume (more likely to be liquid)
        if len(valid_stocks) > max_stocks:
            avg_volumes = self.data[self.data['tic'].isin(valid_stocks)].groupby('tic')['volume'].mean()
            valid_stocks = avg_volumes.nlargest(max_stocks).index.tolist()

        self.data = self.data[self.data['tic'].isin(valid_stocks)]
        self.unique_dates = sorted(self.data['date'].unique())
        self.tickers = sorted(self.data['tic'].unique())
        self.num_stocks = len(self.tickers)
        self.lookback = lookback_window
        self.num_features = 9  # Updated with new features (BB_upper, BB_lower)

        # Create ticker lookup for faster access
        self.ticker_indices = {ticker: i for i, ticker in enumerate(self.tickers)}

        self._add_technical_indicators()
        self._scale_features()

        # Store data in a more efficient format
        self.prepare_training_data()

        print(f"Loaded {self.num_stocks} stocks with {len(self.unique_dates)} trading days")
        print(f"Date range: {self.unique_dates[0]} to {self.unique_dates[-1]}")

    def _add_technical_indicators(self):
        print("Adding technical indicators...")
        indicators = ['Returns', 'Volume_MA', 'Price_MA', 'RSI', 'MACD', 'Volatility', 'Momentum', 'BB_upper', 'BB_lower']

        # Process one ticker at a time to avoid memory issues
        for tic in tqdm(self.tickers, desc="Processing tickers"):
            mask = self.data['tic'] == tic
            stock_data = self.data.loc[mask].copy()

            # Calculate technical indicators
            stock_data['Returns'] = stock_data['close'].pct_change()
            stock_data['Price_MA'] = stock_data['close'].rolling(window=20, min_periods=1).mean()
            stock_data['Momentum'] = stock_data['close'].pct_change(periods=10)
            stock_data['Volume_MA'] = stock_data['volume'].rolling(window=10, min_periods=1).mean()

            # RSI Calculation
            delta = stock_data['close'].diff()
            gain = (delta.clip(lower=0)).rolling(window=14, min_periods=1).mean()
            loss = (-delta.clip(upper=0)).rolling(window=14, min_periods=1).mean()
            rs = gain / (loss + 1e-8)
            stock_data['RSI'] = 100 - (100 / (1 + rs))

            # MACD Calculation
            exp1 = stock_data['close'].ewm(span=12, adjust=False).mean()
            exp2 = stock_data['close'].ewm(span=26, adjust=False).mean()
            stock_data['MACD'] = exp1 - exp2

            # Volatility
            stock_data['Volatility'] = stock_data['Returns'].rolling(window=30, min_periods=1).std()

            # Bollinger Bands
            stock_data['BB_upper'] = stock_data['Price_MA'] + (stock_data['Volatility'] * 2)
            stock_data['BB_lower'] = stock_data['Price_MA'] - (stock_data['Volatility'] * 2)

            # Update the main dataframe
            self.data.loc[mask, indicators] = stock_data[indicators]

        # Fill missing values
        self.data = self.data.fillna(0)
        print("Technical indicators added successfully")

    def _scale_features(self):
        print("Scaling features...")
        feature_columns = ['Returns', 'Volume_MA', 'Price_MA', 'RSI', 'MACD', 'Volatility', 'Momentum', 'BB_upper', 'BB_lower']

        for tic in tqdm(self.tickers, desc="Scaling features"):
            mask = self.data['tic'] == tic
            scaler = StandardScaler()
            self.data.loc[mask, feature_columns] = scaler.fit_transform(self.data.loc[mask, feature_columns])

    def prepare_training_data(self):
        print("Preparing training data...")
        # Pre-allocate arrays for features and prices
        self.all_features = {}
        self.all_prices = {}
        feature_columns = ['Returns', 'Volume_MA', 'Price_MA', 'RSI', 'MACD', 'Volatility', 'Momentum', 'BB_upper', 'BB_lower']

        # Process data by date for efficient access during training
        for date_idx, date in enumerate(tqdm(self.unique_dates, desc="Processing dates")):
            date_mask = self.data['date'] == date
            date_data = self.data[date_mask]

            features = np.zeros((self.num_stocks, self.num_features))
            prices = np.zeros(self.num_stocks)

            for _, row in date_data.iterrows():
                ticker_idx = self.ticker_indices.get(row['tic'])
                if ticker_idx is not None:
                    features[ticker_idx] = row[feature_columns].values
                    prices[ticker_idx] = row['close']

            self.all_features[date] = features
            self.all_prices[date] = prices

    def __iter__(self):
        """Initialize iterator"""
        self.current_idx = self.lookback
        return self

    def __next__(self):
        """Get next batch of data"""
        if self.current_idx >= len(self.unique_dates):
            raise StopIteration

        current_date = self.unique_dates[self.current_idx]
        window_dates = self.unique_dates[self.current_idx - self.lookback:self.current_idx]

        # Get lookback window features
        features = np.zeros((self.num_stocks, self.lookback, self.num_features), dtype=np.float32)

        for i, date in enumerate(window_dates):
            if date in self.all_features:
                features[:, i, :] = self.all_features[date]

        prices = self.all_prices[current_date]

        self.current_idx += 1
        return {
            'features': features,
            'prices': prices,
            'date': current_date,
            'tickers': self.tickers
        }

    def reset(self):
        """Reset the iterator"""
        self.current_idx = self.lookback
        return self

In [4]:
class PortfolioTradingEnv(gym.Env):
    def __init__(self, data_iterator, initial_cash=10000, transaction_cost=0.001):
        super().__init__()
        self.data_iterator = data_iterator
        self.initial_cash = initial_cash
        self.transaction_cost = transaction_cost
        self.num_stocks = data_iterator.num_stocks
        self.lookback = data_iterator.lookback
        self.num_features = data_iterator.num_features
        self.episode_reward = 0  # Track total reward
        self.episode_steps = 0  # Track number of steps
        self.debug_info = {}  # Store debug information

        # Action space: [cash allocation, stock1 allocation, ..., stockN allocation]
        self.action_space = spaces.Box(
            low=0,
            high=1,
            shape=(self.num_stocks + 1,),
            dtype=np.float32
        )

        # Observation space: (num_stocks, lookback, num_features)
        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(self.num_stocks, self.lookback, self.num_features),
            dtype=np.float32
        )

        self.reset()

    def reset(self, seed=None, options=None):
        """Reset the environment"""
        # Reset the data iterator
        self.data_iterator.reset()
        self.cash = self.initial_cash
        self.holdings = np.zeros(self.num_stocks)
        self.episode_reward = 0  # Reset total reward
        self.episode_steps = 0  # Reset step counter
        self.debug_info = {}  # Reset debug info

        try:
            self.current_step = next(self.data_iterator)
            return self._get_observation(), {}
        except StopIteration:
            # Handle the case where there's no more data
            return np.zeros((self.num_stocks, self.lookback, self.num_features), dtype=np.float32), {}

    def _get_observation(self):
        """Get the current observation"""
        return self.current_step['features'].astype(np.float32)

    def step(self, action):
      """Take a step in the environment"""
      self.episode_steps += 1

      # Handle case where action is a scalar (possible in vector environments)
      if np.isscalar(action):
        action = np.array([action])  # Convert to array with single element

      # Ensure action has correct shape
      if action.shape != (self.num_stocks + 1,):
        if len(action.shape) > 1:
            # If action is a batch (from vector env), take the first one
            action = action[0]
        else:
            # If action is still wrong shape, create a default action (all cash)
            print(f"Warning: Invalid action shape {action.shape}, expected {(self.num_stocks + 1,)}")
            action = np.zeros(self.num_stocks + 1)
            action[0] = 1.0  # All cash

      # Normalize action
      action = np.clip(action, 0, 1)
      action_sum = action.sum()
      if action_sum > 0:
        action /= action_sum
      else:
        # Handle the case where all actions are zero
        action[0] = 1.0  # Set cash allocation to 100%

      # Store action for debugging
      if self.episode_steps % 100 == 0:
        self.debug_info['action'] = action.copy()

      current_prices = self.current_step['prices']
      portfolio_value = self.cash + np.sum(self.holdings * current_prices)

      # Calculate target allocations
      target_values = action[1:] * portfolio_value
      current_values = self.holdings * current_prices
      delta_values = target_values - current_values

      # Apply transaction costs
      transaction_costs = np.abs(delta_values).sum() * self.transaction_cost

      # Update holdings and cash
      for i in range(self.num_stocks):
        if current_prices[i] > 0:  # Avoid division by zero
            self.holdings[i] += delta_values[i] / current_prices[i]

      self.cash = portfolio_value - np.sum(self.holdings * current_prices) - transaction_costs

      # Store current portfolio value for reward calculation
      old_portfolio_value = portfolio_value

      try:
        # Move to next time step
        self.current_step = next(self.data_iterator)
        new_prices = self.current_step['prices']
        done = False
      except StopIteration:
        new_prices = current_prices  # Use current prices if no more data
        done = True

      # Calculate reward (daily return)
      new_portfolio_value = self.cash + np.sum(self.holdings * new_prices)
      reward = (new_portfolio_value / old_portfolio_value) - 1

      # Apply penalty for excessive trading
      reward -= transaction_costs / old_portfolio_value

      # Store debugging information
      if self.episode_steps % 100 == 0 or done:
        self.debug_info.update({
            'portfolio_value_before': old_portfolio_value,
            'portfolio_value_after': new_portfolio_value,
            'transaction_costs': transaction_costs,
            'reward': reward,
            'cash': self.cash,
            'holdings_sum': np.sum(self.holdings * new_prices)
        })
        print(f"Step {self.episode_steps}: Reward = {reward:.6f}, "
              f"Portfolio Value: {new_portfolio_value:.2f}, "
              f"Transaction Costs: {transaction_costs:.2f}")

      # Update total reward
      self.episode_reward += float(reward)

      return (
        self._get_observation() if not done else np.zeros_like(self.observation_space.sample()),
        float(reward),
        done,
        False,
        {
            "portfolio_value": new_portfolio_value,
            "total_reward": self.episode_reward,
            "transaction_costs": transaction_costs,
            "cash": self.cash,
            "holdings_value": np.sum(self.holdings * new_prices)
        }
    )

In [5]:
class PortfolioRecommender:
    def __init__(self, model_path, data_path, max_stocks=1568, lookback=30):
        self.model = PPO.load(model_path)
        self.max_stocks = max_stocks
        self.lookback = lookback

        # Load the most recent data for prediction
        data = pd.read_csv(data_path)
        data['date'] = pd.to_datetime(data['date'])

        # Get the most recent date
        self.latest_date = data['date'].max()

        # Select top stocks by volume (same filtering as in training)
        recent_data = data[data['date'] >= (self.latest_date - pd.DateOffset(days=30))]
        avg_volumes = recent_data.groupby('tic')['volume'].mean()
        top_tickers = avg_volumes.nlargest(max_stocks).index.tolist()

        # Get the list of tickers (must match the order used in training)
        self.tickers = sorted(top_tickers)

        # Store the latest prices
        latest_data = data[data['date'] == self.latest_date]
        self.latest_prices = {row['tic']: row['close'] for _, row in latest_data.iterrows()
                              if row['tic'] in self.tickers}

        # Add technical indicators for better prediction
        self._prepare_features(data)

    def _prepare_features(self, data):
        # Filter for only the tickers we're using
        filtered_data = data[data['tic'].isin(self.tickers)]

        # Get the most recent dates for feature calculation
        recent_dates = sorted(filtered_data['date'].unique())[-self.lookback:]
        recent_data = filtered_data[filtered_data['date'].isin(recent_dates)]

        # Calculate features (matching the training features)
        features = np.zeros((self.max_stocks, self.lookback, 9), dtype=np.float32)

        for i, ticker in enumerate(self.tickers):
            if i >= self.max_stocks:
                break

            ticker_data = recent_data[recent_data['tic'] == ticker].sort_values('date')
            if len(ticker_data) == self.lookback:
                # Calculate all features to match training data
                returns = ticker_data['close'].pct_change().fillna(0).values
                volume_ma = ticker_data['volume'].rolling(window=10, min_periods=1).mean().values
                price_ma = ticker_data['close'].rolling(window=20, min_periods=1).mean().values

                # RSI calculation
                delta = ticker_data['close'].diff().fillna(0)
                gain = (delta.clip(lower=0)).rolling(window=14, min_periods=1).mean()
                loss = (-delta.clip(upper=0)).rolling(window=14, min_periods=1).mean()
                rs = gain / (loss + 1e-8)
                rsi = 100 - (100 / (1 + rs)).values

                # MACD
                exp1 = ticker_data['close'].ewm(span=12, adjust=False).mean()
                exp2 = ticker_data['close'].ewm(span=26, adjust=False).mean()
                macd = (exp1 - exp2).values

                # Volatility
                volatility = ticker_data['close'].pct_change().rolling(window=30, min_periods=1).std().fillna(0).values

                # Momentum
                momentum = ticker_data['close'].pct_change(periods=10).fillna(0).values

                # Bollinger Bands
                bb_upper = price_ma + (volatility * 2)
                bb_lower = price_ma - (volatility * 2)

                # Combine all features
                for j in range(self.lookback):
                    features[i, j, 0] = returns[j]
                    features[i, j, 1] = volume_ma[j]
                    features[i, j, 2] = price_ma[j]
                    features[i, j, 3] = rsi[j]
                    features[i, j, 4] = macd[j]
                    features[i, j, 5] = volatility[j]
                    features[i, j, 6] = momentum[j]
                    features[i, j, 7] = bb_upper[j]
                    features[i, j, 8] = bb_lower[j]

        self.recent_features = features

    def recommend_portfolio(self, amount_cad):
        # Use the precomputed features for prediction
        action, _ = self.model.predict(self.recent_features, deterministic=True)

        # Normalize allocations
        action = np.clip(action, 0, 1)
        action_sum = action.sum()
        if action_sum > 0:
            action /= action_sum
        else:
            # Handle the case where all actions are zero
            action[0] = 1.0  # Set cash allocation to 100%

        # Allocate cash based on model recommendation
        allocations = {}
        cash_allocation = action[0] * amount_cad

        # Process only up to the number of tickers we have
        for i, ticker in enumerate(self.tickers):
            if i >= len(self.tickers) or i+1 >= len(action):
                continue

            allocation = action[i+1] * amount_cad
            if allocation > 0 and ticker in self.latest_prices:
                price = self.latest_prices[ticker]
                shares = allocation / price if price > 0 else 0

                allocations[ticker] = {
                    "allocation_percent": float(action[i+1] * 100),
                    "allocation_cad": float(allocation),
                    "price_per_share": float(price),
                    "shares": float(shares)
                }

        # Print some allocation statistics for verification
        print(f"\nAllocation Summary:")
        print(f"Cash: {action[0]*100:.2f}%")
        print(f"Stocks: {(1-action[0])*100:.2f}%")
        print(f"Number of stocks allocated: {len(allocations)}")

        if len(allocations) > 0:
            allocations_array = np.array([details['allocation_percent'] for details in allocations.values()])
            print(f"Max allocation: {np.max(allocations_array):.2f}%")
            print(f"Min allocation: {np.min(allocations_array):.2f}%")
            print(f"Avg allocation: {np.mean(allocations_array):.2f}%")

        return {
            "cash_percent": float(action[0] * 100),
            "cash_amount": float(cash_allocation),
            "stock_allocations": allocations,
            "total_amount": float(amount_cad),
            "recommendation_date": str(self.latest_date)
        }

In [6]:
def evaluate_model(model, data_path, max_stocks=1568, lookback=30, episodes=10):
    """Evaluate the model on a validation dataset"""
    print(f"Evaluating model performance...")

    eval_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=5  # Use more recent data for evaluation
    )

    eval_env = PortfolioTradingEnv(eval_iter)

    total_rewards = []
    portfolio_values = []

    for ep in range(episodes):
        obs, _ = eval_env.reset()
        done = False
        episode_reward = 0
        initial_value = eval_env.initial_cash
        step_count = 0

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _, info = eval_env.step(action)
            episode_reward += reward
            step_count += 1

            if done:
                final_value = info["portfolio_value"]
                portfolio_values.append(final_value)

                # Calculate annualized return
                if step_count > 0:
                    days = step_count
                    annualized_return = ((final_value / initial_value) ** (252 / days) - 1) * 100
                else:
                    annualized_return = 0

                print(f"Episode {ep+1}: Return = {episode_reward:.4f}, "
                      f"Steps = {step_count}, "
                      f"Final Value = ${final_value:.2f}, "
                      f"Annualized Return = {annualized_return:.2f}%")

        total_rewards.append(episode_reward)

    # Calculate summary statistics
    avg_reward = np.mean(total_rewards)
    avg_final_value = np.mean(portfolio_values)
    success_rate = np.sum(np.array(total_rewards) > 0) / len(total_rewards) * 100

    print(f"\nEvaluation Results (over {episodes} episodes):")
    print(f"Average Total Reward: {avg_reward:.4f}")
    print(f"Average Final Portfolio Value: ${avg_final_value:.2f}")
    print(f"Success Rate (positive return): {success_rate:.2f}%")

    return total_rewards, portfolio_values

In [7]:
def train_model(data_path, model_save_path, max_stocks=1568, lookback=30, timesteps=10000):
    print(f"Starting training at {datetime.now()}")

    # Initialize components with more manageable parameters
    train_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=20
    )

    def make_env():
        return PortfolioTradingEnv(train_iter)

    train_env = DummyVecEnv([make_env])

    # Configure PPO model with more appropriate hyperparameters
    model = PPO(
        "MlpPolicy",
        train_env,
        learning_rate=3e-4,
        n_steps=1024,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        ent_coef=0.01,  # Encourage exploration
        verbose=1,
        device="auto",  # Use GPU if available
        tensorboard_log="./ppo_portfolio_tensorboard/"  # Add tensorboard logging
    )

    # Log initial random policy performance - with proper handling of vector environment
    print("Evaluating random policy before training...")
    try:
        obs = train_env.reset()
        done = np.array([False])
        total_reward = 0
        step_count = 0
        max_steps = 100  # Limit evaluation to avoid infinite loops

        while not done.any() and step_count < max_steps:
            # Sample properly shaped action from the action space
            actions = []
            for i in range(train_env.num_envs):
                # Generate an action vector of the correct shape (num_stocks + 1)
                action = np.random.random(train_iter.num_stocks + 1)
                # Normalize to sum to 1
                action = action / action.sum()
                actions.append(action)

            actions = np.array(actions)
            obs, rewards, done, info = train_env.step(actions)
            total_reward += rewards[0]  # Just take the first environment's reward
            step_count += 1

        print(f"Random policy evaluation: Total steps: {step_count}, Total reward: {total_reward}")
    except Exception as e:
        print(f"Error during random policy evaluation: {e}")
        print("Continuing with training anyway...")

    # Start training with progress monitoring
    print(f"Training model with {train_iter.num_stocks} stocks, each with {lookback} days of history")

    # Train in multiple stages with evaluation
    stage_timesteps = timesteps // 5
    for stage in range(5):
        print(f"\nTraining stage {stage+1}/5 ({stage_timesteps} timesteps)...")
        try:
            model.learn(total_timesteps=stage_timesteps, progress_bar=True, reset_num_timesteps=False)

            # Save checkpoint
            checkpoint_path = f"{model_save_path}_checkpoint_{stage+1}"
            model.save(checkpoint_path)
            print(f"Checkpoint saved to {checkpoint_path}")

            # Quick evaluation
            if stage < 4:  # Skip final evaluation as we'll do a full one later
                print("Quick evaluation of current policy...")
                eval_env = make_env()
                for i in range(3):  # Just a quick check with 3 episodes
                    obs, _ = eval_env.reset()
                    done = False
                    episode_reward = 0
                    step_count = 0
                    max_eval_steps = 100  # Limit evaluation steps

                    while not done and step_count < max_eval_steps:
                        action, _ = model.predict(obs, deterministic=True)
                        obs, reward, done, _, info = eval_env.step(action)
                        episode_reward += reward
                        step_count += 1

                    print(f"  Episode {i+1} reward: {episode_reward:.4f}, Steps: {step_count}, "
                          f"Final portfolio: ${info['portfolio_value']:.2f}")
        except Exception as e:
            print(f"Error during training stage {stage+1}: {e}")
            print("Attempting to continue with next stage...")

    # Save the final trained model
    try:
        model.save(model_save_path)
        print(f"Training completed at {datetime.now()} and model saved to {model_save_path}!")
    except Exception as e:
        print(f"Error saving model: {e}")
        print("Training completed but model could not be saved.")

    return model

In [8]:
def calculate_portfolio_metrics(portfolio_values, risk_free_rate=0.02):
    """Calculate common portfolio performance metrics"""
    if len(portfolio_values) < 2:
        return {
            "total_return": 0,
            "sharpe_ratio": 0,
            "max_drawdown": 0,
            "volatility": 0
        }

    # Calculate returns
    returns = np.array([(portfolio_values[i] / portfolio_values[i-1]) - 1
                       for i in range(1, len(portfolio_values))])

    # Calculate metrics
    total_return = (portfolio_values[-1] / portfolio_values[0]) - 1
    daily_returns_mean = np.mean(returns)
    daily_returns_std = np.std(returns)

    # Annualize (assuming 252 trading days)
    annualized_return = ((1 + daily_returns_mean) ** 252) - 1
    annualized_vol = daily_returns_std * np.sqrt(252)

    # Sharpe ratio
    sharpe_ratio = (annualized_return - risk_free_rate) / annualized_vol if annualized_vol > 0 else 0

    # Maximum drawdown
    peak = portfolio_values[0]
    max_drawdown = 0

    for value in portfolio_values:
        if value > peak:
            peak = value
        drawdown = (peak - value) / peak
        max_drawdown = max(max_drawdown, drawdown)

    return {
        "total_return": total_return * 100,  # Convert to percentage
        "annualized_return": annualized_return * 100,
        "volatility": annualized_vol * 100,
        "sharpe_ratio": sharpe_ratio,
        "max_drawdown": max_drawdown * 100,
        "win_rate": np.mean(returns > 0) * 100
    }



In [9]:
def backtest_portfolio(model, data_path, max_stocks=1568, lookback=30, initial_cash=10000, years=1):
    """Run a comprehensive backtest of the portfolio strategy"""
    print(f"Running {years}-year backtest simulation...")

    # Initialize the environment with the backtest data
    backtest_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=years
    )

    backtest_env = PortfolioTradingEnv(backtest_iter, initial_cash=initial_cash)

    # Run backtest
    obs, _ = backtest_env.reset()
    done = False

    portfolio_values = [initial_cash]
    returns = []
    cash_allocations = []
    transaction_costs = []
    holdings_diversification = []
    daily_turnover = []

    step = 0
    while not done:
        action, _ = model.predict(obs, deterministic=True)

        # Track portfolio composition
        cash_allocations.append(action[0])

        # Number of stocks with allocation > 1%
        significant_holdings = sum(1 for a in action[1:] if a > 0.01)
        holdings_diversification.append(significant_holdings)

        # Execute step
        obs, reward, done, _, info = backtest_env.step(action)

        # Track metrics
        portfolio_values.append(info["portfolio_value"])
        if step > 0:
            returns.append((portfolio_values[-1] / portfolio_values[-2]) - 1)
        transaction_costs.append(info["transaction_costs"])

        # Calculate turnover (sum of absolute changes in allocation)
        if step == 0:
            daily_turnover.append(0)
        else:
            turnover = info["transaction_costs"] / (portfolio_values[-2] * backtest_env.transaction_cost)
            daily_turnover.append(turnover)

        step += 1

        # Print progress
        if step % 50 == 0:
            print(f"Backtest step {step}, Portfolio value: ${portfolio_values[-1]:.2f}")

    # Calculate performance metrics
    metrics = calculate_portfolio_metrics(portfolio_values)

    backtest_results = {
        "portfolio_values": portfolio_values,
        "returns": returns,
        "cash_allocations": cash_allocations,
        "transaction_costs": transaction_costs,
        "holdings_diversification": holdings_diversification,
        "daily_turnover": daily_turnover,
        "metrics": metrics,
        "steps": step,
        "final_value": portfolio_values[-1]
    }

    return backtest_results

In [10]:
if __name__ == "__main__":
    import sys
    import os
    import json
    from datetime import datetime

    # Default parameters
    DATA_PATH = "/content/historical_data.csv"
    MODEL_SAVE_PATH = "portfolio_ppo_model"
    MAX_STOCKS = 1568  # Limit number of stocks to make training feasible
    LOOKBACK = 30     # Shorter lookback period
    TIMESTEPS = 10000  # Increased number of training steps
    INITIAL_CASH = 10000
    MODE = "train_and_evaluate"  # Default mode

    # Check if running in Jupyter/Colab environment
    is_notebook = 'ipykernel' in sys.modules

    if is_notebook:
        # For Jupyter/Colab, don't use argparse
        # You can modify these values directly in the notebook
        print("Running in notebook environment. Using default parameters.")
        print(f"DATA_PATH: {DATA_PATH}")
        print(f"MODEL_SAVE_PATH: {MODEL_SAVE_PATH}")
        print(f"MAX_STOCKS: {MAX_STOCKS}")
        print(f"LOOKBACK: {LOOKBACK}")
        print(f"TIMESTEPS: {TIMESTEPS}")
        print(f"INITIAL_CASH: {INITIAL_CASH}")
        print(f"MODE: {MODE}")

        # If you want to change parameters, do it here
        # Example: MODE = "recommend"
    else:
        # For command line execution, use argparse
        import argparse
        parser = argparse.ArgumentParser(description="Portfolio Optimization with Reinforcement Learning")
        parser.add_argument("--data_path", type=str, default=DATA_PATH, help="Path to historical data CSV")
        parser.add_argument("--model_path", type=str, default=MODEL_SAVE_PATH, help="Path to save/load model")
        parser.add_argument("--max_stocks", type=int, default=MAX_STOCKS, help="Maximum number of stocks to consider")
        parser.add_argument("--lookback", type=int, default=LOOKBACK, help="Lookback window size")
        parser.add_argument("--timesteps", type=int, default=TIMESTEPS, help="Number of training timesteps")
        parser.add_argument("--initial_cash", type=float, default=INITIAL_CASH, help="Initial cash for portfolio")
        parser.add_argument("--mode", type=str, default=MODE,
                            choices=["train", "evaluate", "backtest", "recommend", "train_and_evaluate"],
                            help="Operation mode")

        args = parser.parse_args()

        # Update variables from command line args
        DATA_PATH = args.data_path
        MODEL_SAVE_PATH = args.model_path
        MAX_STOCKS = args.max_stocks
        LOOKBACK = args.lookback
        TIMESTEPS = args.timesteps
        INITIAL_CASH = args.initial_cash
        MODE = args.mode

    # Create results directory
    os.makedirs("results", exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Main execution logic
    if MODE == "train" or MODE == "train_and_evaluate":
        print(f"\n{'='*80}\nSTARTING TRAINING\n{'='*80}")
        model = train_model(
            DATA_PATH,
            MODEL_SAVE_PATH,
            MAX_STOCKS,
            LOOKBACK,
            TIMESTEPS
        )
        print(f"Model saved to {MODEL_SAVE_PATH}")
    else:
        # Load existing model
        try:
            from stable_baselines3 import PPO
            print(f"Loading model from {MODEL_SAVE_PATH}")
            model = PPO.load(MODEL_SAVE_PATH)
        except Exception as e:
            print(f"Error loading model: {e}")
            print("Please train a model first or provide a valid model path.")
            exit(1)

    if MODE == "evaluate" or MODE == "train_and_evaluate":
        print(f"\n{'='*80}\nEVALUATING MODEL\n{'='*80}")
        # Evaluate the model
        rewards, values = evaluate_model(
            model,
            DATA_PATH,
            MAX_STOCKS,
            LOOKBACK,
            episodes=10
        )

        # Save evaluation results
        eval_results = {
            "rewards": [float(r) for r in rewards],
            "final_values": [float(v) for v in values],
            "avg_reward": float(np.mean(rewards)),
            "avg_final_value": float(np.mean(values)),
            "max_reward": float(np.max(rewards)),
            "min_reward": float(np.min(rewards)),
            "success_rate": float(np.mean(np.array(rewards) > 0) * 100)
        }

        with open(f"results/evaluation_{timestamp}.json", "w") as f:
            json.dump(eval_results, f, indent=2)

        print(f"Evaluation results saved to results/evaluation_{timestamp}.json")

    if MODE == "backtest":
        print(f"\n{'='*80}\nRUNNING BACKTEST\n{'='*80}")
        # Run comprehensive backtest
        backtest_results = backtest_portfolio(
            model,
            DATA_PATH,
            MAX_STOCKS,
            LOOKBACK,
            INITIAL_CASH,
            years=2  # Use 2 years of data for backtest
        )

        # Print backtest summary
        metrics = backtest_results["metrics"]
        print("\nBacktest Summary:")
        print(f"Total Return: {metrics['total_return']:.2f}%")
        print(f"Annualized Return: {metrics['annualized_return']:.2f}%")
        print(f"Volatility: {metrics['volatility']:.2f}%")
        print(f"Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
        print(f"Maximum Drawdown: {metrics['max_drawdown']:.2f}%")
        print(f"Win Rate: {metrics['win_rate']:.2f}%")
        print(f"Average Cash Allocation: {np.mean(backtest_results['cash_allocations']) * 100:.2f}%")
        print(f"Average Number of Holdings: {np.mean(backtest_results['holdings_diversification']):.1f} stocks")
        print(f"Average Daily Turnover: {np.mean(backtest_results['daily_turnover']) * 100:.2f}%")

        # Save backtest results (excluding large arrays)
        backtest_summary = {
            "initial_cash": INITIAL_CASH,
            "final_value": float(backtest_results["final_value"]),
            "trading_days": backtest_results["steps"],
            "metrics": {k: float(v) for k, v in metrics.items()},
            "avg_cash_allocation": float(np.mean(backtest_results["cash_allocations"]) * 100),
            "avg_holdings": float(np.mean(backtest_results["holdings_diversification"])),
            "avg_turnover": float(np.mean(backtest_results["daily_turnover"]) * 100),
            "avg_transaction_costs": float(np.mean(backtest_results["transaction_costs"]))
        }

        with open(f"results/backtest_{timestamp}.json", "w") as f:
            json.dump(backtest_summary, f, indent=2)

        print(f"Backtest summary saved to results/backtest_{timestamp}.json")

    if MODE == "recommend":
        print(f"\n{'='*80}\nGENERATING PORTFOLIO RECOMMENDATION\n{'='*80}")
        # Generate portfolio recommendation
        recommender = PortfolioRecommender(
            MODEL_SAVE_PATH,
            DATA_PATH,
            max_stocks=MAX_STOCKS,
            lookback=LOOKBACK
        )

        recommendation = recommender.recommend_portfolio(INITIAL_CASH)

        # Print recommendation
        print(f"Recommended portfolio allocation (as of {recommendation['recommendation_date']}):")
        print(f"Cash: ${recommendation['cash_amount']:.2f} ({recommendation['cash_percent']:.2f}%)")
        print("\nStock allocations:")

        # Sort allocations by percentage (largest first)
        sorted_allocations = sorted(
            recommendation['stock_allocations'].items(),
            key=lambda x: x[1]['allocation_percent'],
            reverse=True
        )

        for ticker, details in sorted_allocations:
            if details['allocation_percent'] > 1.0:  # Only show significant allocations
                print(f"{ticker}: ${details['allocation_cad']:.2f} "
                      f"({details['allocation_percent']:.2f}%) - "
                      f"{details['shares']:.4f} shares @ ${details['price_per_share']:.2f}")

        # Calculate some portfolio statistics
        if sorted_allocations:
            allocations = [details['allocation_percent'] for _, details in sorted_allocations]
            print("\nPortfolio allocation statistics:")
            print(f"Number of stocks: {len(allocations)}")
            print(f"Max allocation: {max(allocations):.2f}%")
            print(f"Min allocation: {min(allocations):.2f}%")
            print(f"Avg allocation: {sum(allocations)/len(allocations):.2f}%")
            print(f"Diversification score: {len([a for a in allocations if a > 1.0])}")

        # Save recommendation
        with open(f"results/recommendation_{timestamp}.json", "w") as f:
            json.dump(recommendation, f, indent=2)

        print(f"Recommendation saved to results/recommendation_{timestamp}.json")

    print(f"\n{'='*80}\nPORTFOLIO OPTIMIZATION COMPLETE\n{'='*80}")
    print(f"Execution time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"All results saved in the 'results' directory.")

Running in notebook environment. Using default parameters.
DATA_PATH: /content/historical_data.csv
MODEL_SAVE_PATH: portfolio_ppo_model
MAX_STOCKS: 1568
LOOKBACK: 30
TIMESTEPS: 10000
INITIAL_CASH: 10000
MODE: train_and_evaluate

STARTING TRAINING
Starting training at 2025-03-19 15:32:35.576527
Loading data from /content/historical_data.csv...
Adding technical indicators...


Processing tickers: 100%|██████████| 1568/1568 [06:40<00:00,  3.91it/s]


Technical indicators added successfully
Scaling features...


Scaling features: 100%|██████████| 1568/1568 [10:16<00:00,  2.54it/s]


Preparing training data...


...

Technical indicators added successfully
Scaling features...


Scaling features: 100%|██████████| 1568/1568 [04:29<00:00,  5.82it/s]


Preparing training data...


Processing dates: 100%|██████████| 1256/1256 [08:51<00:00,  2.36it/s]


Loaded 1568 stocks with 1256 trading days
Date range: 2019-12-30 00:00:00 to 2024-12-30 00:00:00
Step 100: Reward = 0.015398, Portfolio Value: 9871.51, Transaction Costs: 2.34
Step 200: Reward = 0.002281, Portfolio Value: 10902.88, Transaction Costs: 5.01
Step 300: Reward = -0.003241, Portfolio Value: 12476.98, Transaction Costs: 4.60
Step 400: Reward = 0.003989, Portfolio Value: 12780.31, Transaction Costs: 3.60
Step 500: Reward = 0.002051, Portfolio Value: 12480.59, Transaction Costs: 3.36
Step 600: Reward = 0.005521, Portfolio Value: 11100.56, Transaction Costs: 1.94
Step 700: Reward = 0.001986, Portfolio Value: 11232.60, Transaction Costs: 5.25
Step 800: Reward = -0.002854, Portfolio Value: 11368.52, Transaction Costs: 3.33
Step 900: Reward = -0.001259, Portfolio Value: 11204.86, Transaction Costs: 1.91
Step 1000: Reward = 0.007882, Portfolio Value: 11375.26, Transaction Costs: 0.74
Step 1100: Reward = 0.002215, Portfolio Value: 12287.09, Transaction Costs: 0.30
Step 1200: Reward =

In [11]:
# Change the mode to 'recommend'
MODE = "recommend"

# Optionally, you can modify the initial cash amount
INITIAL_CASH = 10000  # or any other amount you prefer

# Run the recommendation part of the code
if MODE == "recommend":
    print(f"\n{'='*80}\nGENERATING PORTFOLIO RECOMMENDATION\n{'='*80}")
    # Generate portfolio recommendation
    recommender = PortfolioRecommender(
        MODEL_SAVE_PATH,
        DATA_PATH,
        max_stocks=MAX_STOCKS,
        lookback=LOOKBACK
    )

    recommendation = recommender.recommend_portfolio(INITIAL_CASH)

    # Print recommendation
    print(f"Recommended portfolio allocation (as of {recommendation['recommendation_date']}):")
    print(f"Cash: ${recommendation['cash_amount']:.2f} ({recommendation['cash_percent']:.2f}%)")
    print("\nStock allocations:")

    # Sort allocations by percentage (largest first)
    sorted_allocations = sorted(
        recommendation['stock_allocations'].items(),
        key=lambda x: x[1]['allocation_percent'],
        reverse=True
    )

    for ticker, details in sorted_allocations:
        if details['allocation_percent'] > 1.0:  # Only show significant allocations
            print(f"{ticker}: ${details['allocation_cad']:.2f} "
                  f"({details['allocation_percent']:.2f}%) - "
                  f"{details['shares']:.4f} shares @ ${details['price_per_share']:.2f}")

    # Calculate some portfolio statistics
    if sorted_allocations:
        allocations = [details['allocation_percent'] for _, details in sorted_allocations]
        print("\nPortfolio allocation statistics:")
        print(f"Number of stocks: {len(allocations)}")
        print(f"Max allocation: {max(allocations):.2f}%")
        print(f"Min allocation: {min(allocations):.2f}%")
        print(f"Avg allocation: {sum(allocations)/len(allocations):.2f}%")
        print(f"Diversification score: {len([a for a in allocations if a > 1.0])}")

    # Save recommendation
    with open(f"results/recommendation_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json", "w") as f:
        import json
        json.dump(recommendation, f, indent=2)

    print(f"Recommendation saved to results directory")


GENERATING PORTFOLIO RECOMMENDATION

Allocation Summary:
Cash: 0.00%
Stocks: 100.00%
Number of stocks allocated: 766
Max allocation: 0.52%
Min allocation: 0.00%
Avg allocation: 0.13%
Recommended portfolio allocation (as of 2024-12-30 00:00:00):
Cash: $0.00 (0.00%)

Stock allocations:

Portfolio allocation statistics:
Number of stocks: 766
Max allocation: 0.52%
Min allocation: 0.00%
Avg allocation: 0.13%
Diversification score: 0
Recommendation saved to results directory


In [12]:
# Change the mode to 'backtest'
MODE = "backtest"

# Optionally, customize these parameters
INITIAL_CASH = 10000  # Starting portfolio value
LOOKBACK = 30         # Days of history to consider
MAX_STOCKS = 1568      # Maximum number of stocks to consider

# Run the backtest
if MODE == "backtest":
    print(f"\n{'='*80}\nRUNNING BACKTEST\n{'='*80}")
    # Run comprehensive backtest
    backtest_results = backtest_portfolio(
        model,
        DATA_PATH,
        MAX_STOCKS,
        LOOKBACK,
        INITIAL_CASH,
        years=2  # Use 2 years of data for backtest
    )

    # Print backtest summary
    metrics = backtest_results["metrics"]
    print("\nBacktest Summary:")
    print(f"Total Return: {metrics['total_return']:.2f}%")
    print(f"Annualized Return: {metrics['annualized_return']:.2f}%")
    print(f"Volatility: {metrics['volatility']:.2f}%")
    print(f"Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
    print(f"Maximum Drawdown: {metrics['max_drawdown']:.2f}%")
    print(f"Win Rate: {metrics['win_rate']:.2f}%")
    print(f"Average Cash Allocation: {np.mean(backtest_results['cash_allocations']) * 100:.2f}%")
    print(f"Average Number of Holdings: {np.mean(backtest_results['holdings_diversification']):.1f} stocks")
    print(f"Average Daily Turnover: {np.mean(backtest_results['daily_turnover']) * 100:.2f}%")

    # Save backtest results (excluding large arrays)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backtest_summary = {
        "initial_cash": INITIAL_CASH,
        "final_value": float(backtest_results["final_value"]),
        "trading_days": backtest_results["steps"],
        "metrics": {k: float(v) for k, v in metrics.items()},
        "avg_cash_allocation": float(np.mean(backtest_results["cash_allocations"]) * 100),
        "avg_holdings": float(np.mean(backtest_results["holdings_diversification"])),
        "avg_turnover": float(np.mean(backtest_results["daily_turnover"]) * 100),
        "avg_transaction_costs": float(np.mean(backtest_results["transaction_costs"]))
    }

    with open(f"results/backtest_{timestamp}.json", "w") as f:
        import json
        json.dump(backtest_summary, f, indent=2)

    print(f"Backtest summary saved to results/backtest_{timestamp}.json")


RUNNING BACKTEST
Running 2-year backtest simulation...
Loading data from /content/historical_data.csv...
Adding technical indicators...


Processing tickers: 100%|██████████| 1568/1568 [01:32<00:00, 16.91it/s]


Technical indicators added successfully
Scaling features...


Scaling features: 100%|██████████| 1568/1568 [02:04<00:00, 12.62it/s]


Preparing training data...


Processing dates: 100%|██████████| 502/502 [03:47<00:00,  2.21it/s]


Loaded 1568 stocks with 502 trading days
Date range: 2022-12-30 00:00:00 to 2024-12-30 00:00:00
Backtest step 50, Portfolio value: $9843.93
Step 100: Reward = 0.004215, Portfolio Value: 9667.78, Transaction Costs: 0.98
Backtest step 100, Portfolio value: $9667.78
Backtest step 150, Portfolio value: $9716.20
Step 200: Reward = 0.003841, Portfolio Value: 9661.27, Transaction Costs: 1.25
Backtest step 200, Portfolio value: $9661.27
Backtest step 250, Portfolio value: $10029.82
Step 300: Reward = -0.003373, Portfolio Value: 10476.04, Transaction Costs: 5.06
Backtest step 300, Portfolio value: $10476.04
Backtest step 350, Portfolio value: $10718.39
Step 400: Reward = 0.000862, Portfolio Value: 11062.68, Transaction Costs: 2.09
Backtest step 400, Portfolio value: $11062.68
Backtest step 450, Portfolio value: $11952.73
Step 472: Reward = -0.000029, Portfolio Value: 11754.00, Transaction Costs: 0.17

Backtest Summary:
Total Return: 17.54%
Annualized Return: 9.38%
Volatility: 8.20%
Sharpe Ratio

In [13]:
import numpy as np
import pandas as pd
from datetime import datetime
import os
import json
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv
import itertools
import time
from tqdm import tqdm


def hyperparameter_tuning(
    data_path,
    base_model_path="tuning_models",
    max_stocks_options=[50, 100],
    lookback_options=[15, 30, 45],
    learning_rate_options=[1e-4, 3e-4, 1e-3],
    gamma_options=[0.95, 0.99],
    ent_coef_options=[0.01, 0.005],
    n_steps_options=[512, 1024],
    batch_size_options=[64, 128],
    transaction_cost_options=[0.001, 0.002],
    n_epochs_options=[5, 10],
    timesteps=10000,  # Reduced for tuning
    eval_episodes=3,   # Reduced for tuning
    n_eval_envs=2
):
    """
    Perform hyperparameter tuning for the portfolio optimization model.

    Parameters:
    -----------
    data_path : str
        Path to the historical data CSV
    base_model_path : str
        Base path to save trained models
    max_stocks_options : list
        Different numbers of stocks to consider
    lookback_options : list
        Different lookback window sizes
    learning_rate_options : list
        Different learning rates for PPO
    gamma_options : list
        Different discount factors
    ent_coef_options : list
        Different entropy coefficients for exploration
    n_steps_options : list
        Different step sizes for rollout
    batch_size_options : list
        Different batch sizes for training
    transaction_cost_options : list
        Different transaction cost values
    n_epochs_options : list
        Different numbers of optimization epochs
    timesteps : int
        Number of timesteps for each training run
    eval_episodes : int
        Number of episodes for evaluation
    n_eval_envs : int
        Number of environments for parallel evaluation

    Returns:
    --------
    dict
        Results of hyperparameter tuning
    """
    # Create results directory
    os.makedirs(os.path.join(base_model_path, "results"), exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    tuning_results_path = os.path.join(base_model_path, "results", f"tuning_results_{timestamp}.json")

    # Track all results
    all_results = []

    # Generate parameter grid (selecting key parameters to avoid combinatorial explosion)
    parameter_grid = []
    for max_stocks, lookback, lr, gamma, ent_coef in itertools.product(
        max_stocks_options,
        lookback_options,
        learning_rate_options,
        gamma_options,
        ent_coef_options
    ):
        # Fixed parameters for initial tuning round
        n_steps = n_steps_options[0]
        batch_size = batch_size_options[0]
        transaction_cost = transaction_cost_options[0]
        n_epochs = n_epochs_options[0]

        parameter_grid.append({
            "max_stocks": max_stocks,
            "lookback": lookback,
            "learning_rate": lr,
            "gamma": gamma,
            "ent_coef": ent_coef,
            "n_steps": n_steps,
            "batch_size": batch_size,
            "transaction_cost": transaction_cost,
            "n_epochs": n_epochs
        })

    print(f"Tuning with {len(parameter_grid)} parameter combinations")

    # Run tuning
    for i, params in enumerate(tqdm(parameter_grid, desc="Hyperparameter tuning")):
        start_time = time.time()

        try:
            # Extract parameters
            max_stocks = params["max_stocks"]
            lookback = params["lookback"]
            learning_rate = params["learning_rate"]
            gamma = params["gamma"]
            ent_coef = params["ent_coef"]
            n_steps = params["n_steps"]
            batch_size = params["batch_size"]
            transaction_cost = params["transaction_cost"]
            n_epochs = params["n_epochs"]

            model_name = f"model_{i+1}_ms{max_stocks}_lb{lookback}_lr{learning_rate}_g{gamma}_e{ent_coef}"
            model_path = os.path.join(base_model_path, model_name)

            print(f"\nTraining model {i+1}/{len(parameter_grid)}: {model_name}")

            # Initialize data iterator
            train_iter = StockPortfolioIterator(
                data_path,
                lookback_window=lookback,
                min_history=100,
                max_stocks=max_stocks,
                train_years=10  # Reduced for faster tuning
            )

            # Create environment
            def make_env(transaction_cost=transaction_cost):
                return PortfolioTradingEnv(train_iter.reset(), transaction_cost=transaction_cost)

            train_env = DummyVecEnv([make_env])

            # Create model
            model = PPO(
                "MlpPolicy",
                train_env,
                learning_rate=learning_rate,
                n_steps=n_steps,
                batch_size=batch_size,
                n_epochs=n_epochs,
                gamma=gamma,
                ent_coef=ent_coef,
                verbose=0,
                device="auto"
            )

            # Train model with reduced timesteps for tuning
            model.learn(total_timesteps=timesteps, progress_bar=True)

            # Evaluate model
            rewards, values = evaluate_model(
                model,
                data_path,
                max_stocks,
                lookback,
                episodes=eval_episodes,
                verbose=False
            )

            # Run backtest
            backtest_results = backtest_portfolio(
                model,
                data_path,
                max_stocks,
                lookback,
                initial_cash=10000,
                years=1,  # Reduced for faster tuning
                verbose=False
            )

            # Calculate metrics
            train_time = time.time() - start_time
            avg_reward = np.mean(rewards)
            avg_final_value = np.mean(values)
            success_rate = np.mean(np.array(rewards) > 0) * 100

            # Extract key backtest metrics
            backtest_metrics = backtest_results["metrics"]

            # Store results
            result = {
                "model_name": model_name,
                "parameters": params,
                "training_time": train_time,
                "evaluation": {
                    "avg_reward": float(avg_reward),
                    "avg_final_value": float(avg_final_value),
                    "success_rate": float(success_rate)
                },
                "backtest": {
                    "total_return": float(backtest_metrics["total_return"]),
                    "annualized_return": float(backtest_metrics["annualized_return"]),
                    "volatility": float(backtest_metrics["volatility"]),
                    "sharpe_ratio": float(backtest_metrics["sharpe_ratio"]),
                    "max_drawdown": float(backtest_metrics["max_drawdown"]),
                    "win_rate": float(backtest_metrics["win_rate"])
                },
                "portfolio_characteristics": {
                    "avg_cash_allocation": float(np.mean(backtest_results["cash_allocations"]) * 100),
                    "avg_holdings": float(np.mean(backtest_results["holdings_diversification"])),
                    "avg_turnover": float(np.mean(backtest_results["daily_turnover"]) * 100)
                }
            }

            all_results.append(result)

            # Save model
            model.save(model_path)

            # Save interim results
            with open(tuning_results_path, "w") as f:
                json.dump(all_results, f, indent=2)

            print(f"Model {i+1} results:")
            print(f"  Avg reward: {avg_reward:.4f}")
            print(f"  Avg final value: ${avg_final_value:.2f}")
            print(f"  Success rate: {success_rate:.2f}%")
            print(f"  Sharpe ratio: {backtest_metrics['sharpe_ratio']:.2f}")
            print(f"  Annualized return: {backtest_metrics['annualized_return']:.2f}%")
            print(f"  Max drawdown: {backtest_metrics['max_drawdown']:.2f}%")

        except Exception as e:
            print(f"Error in parameter set {i+1}: {e}")
            all_results.append({
                "model_name": f"model_{i+1}",
                "parameters": params,
                "error": str(e)
            })

    # Find best model by different metrics
    valid_results = [r for r in all_results if "error" not in r]

    if valid_results:
        best_by_reward = max(valid_results, key=lambda x: x["evaluation"]["avg_reward"])
        best_by_sharpe = max(valid_results, key=lambda x: x["backtest"]["sharpe_ratio"])
        best_by_return = max(valid_results, key=lambda x: x["backtest"]["annualized_return"])

        best_models = {
            "best_by_reward": best_by_reward,
            "best_by_sharpe": best_by_sharpe,
            "best_by_return": best_by_return
        }

        # Save final results
        final_results = {
            "timestamp": timestamp,
            "all_results": all_results,
            "best_models": best_models
        }

        with open(tuning_results_path, "w") as f:
            json.dump(final_results, f, indent=2)

        print("\nHyperparameter tuning complete!")
        print(f"Results saved to: {tuning_results_path}")

        print("\nBest model by evaluation reward:")
        print_model_summary(best_by_reward)

        print("\nBest model by Sharpe ratio:")
        print_model_summary(best_by_sharpe)

        print("\nBest model by annualized return:")
        print_model_summary(best_by_return)

        return final_results
    else:
        print("No valid results found.")
        return {"timestamp": timestamp, "all_results": all_results}


def print_model_summary(model_result):
    """Print a summary of model performance"""
    params = model_result["parameters"]
    eval_metrics = model_result["evaluation"]
    backtest_metrics = model_result["backtest"]
    portfolio_metrics = model_result["portfolio_characteristics"]

    print(f"Model: {model_result['model_name']}")
    print(f"Parameters: max_stocks={params['max_stocks']}, lookback={params['lookback']}, "
          f"lr={params['learning_rate']}, gamma={params['gamma']}, ent_coef={params['ent_coef']}")
    print(f"Evaluation: reward={eval_metrics['avg_reward']:.4f}, final_value=${eval_metrics['avg_final_value']:.2f}")
    print(f"Backtest: return={backtest_metrics['annualized_return']:.2f}%, "
          f"sharpe={backtest_metrics['sharpe_ratio']:.2f}, "
          f"drawdown={backtest_metrics['max_drawdown']:.2f}%")
    print(f"Portfolio: cash={portfolio_metrics['avg_cash_allocation']:.2f}%, "
          f"holdings={portfolio_metrics['avg_holdings']:.1f}, "
          f"turnover={portfolio_metrics['avg_turnover']:.2f}%")


def fine_tune_best_model(
    data_path,
    base_results_path,
    best_model_type="best_by_sharpe",  # Options: best_by_reward, best_by_sharpe, best_by_return
    timesteps=10000,  # Full training for fine-tuning
    eval_episodes=10
):
    """
    Fine-tune the best model from initial hyperparameter tuning

    Parameters:
    -----------
    data_path : str
        Path to the historical data CSV
    base_results_path : str
        Path to the tuning results JSON file
    best_model_type : str
        Which "best" model to fine-tune
    timesteps : int
        Number of timesteps for fine-tuning
    eval_episodes : int
        Number of episodes for evaluation

    Returns:
    --------
    dict
        Results of fine-tuning
    """
    # Load tuning results
    with open(base_results_path, "r") as f:
        tuning_results = json.load(f)

    best_model = tuning_results["best_models"][best_model_type]
    best_params = best_model["parameters"]

    print(f"Fine-tuning the {best_model_type} model:")
    print_model_summary(best_model)

    # Extract parameters
    max_stocks = best_params["max_stocks"]
    lookback = best_params["lookback"]
    learning_rate = best_params["learning_rate"]
    gamma = best_params["gamma"]
    ent_coef = best_params["ent_coef"]
    n_steps = best_params["n_steps"]
    batch_size = best_params["batch_size"]
    transaction_cost = best_params["transaction_cost"]
    n_epochs = best_params["n_epochs"]

    # Create a timestamp for this fine-tuning run
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_name = f"finetuned_{best_model_type}_{timestamp}"
    model_path = os.path.join(os.path.dirname(base_results_path), model_name)

    # Initialize data iterator with full data
    train_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=20  # Full training data
    )

    # Create environment
    def make_env(transaction_cost=transaction_cost):
        return PortfolioTradingEnv(train_iter.reset(), transaction_cost=transaction_cost)

    train_env = DummyVecEnv([make_env])

    # Create model
    model = PPO(
        "MlpPolicy",
        train_env,
        learning_rate=learning_rate,
        n_steps=n_steps,
        batch_size=batch_size,
        n_epochs=n_epochs,
        gamma=gamma,
        ent_coef=ent_coef,
        verbose=1,
        device="auto",
        tensorboard_log=os.path.join(os.path.dirname(base_results_path), "fine_tuning_tensorboard")
    )

    print(f"\nFine-tuning with {timesteps} timesteps...")

    # Train model with full timesteps
    model.learn(total_timesteps=timesteps, progress_bar=True)
    model.save(model_path)

    print(f"Model saved to {model_path}")

    # Evaluate model
    print("\nEvaluating fine-tuned model...")
    rewards, values = evaluate_model(
        model,
        data_path,
        max_stocks,
        lookback,
        episodes=eval_episodes
    )

    # Run backtest
    print("\nRunning backtest with fine-tuned model...")
    backtest_results = backtest_portfolio(
        model,
        data_path,
        max_stocks,
        lookback,
        initial_cash=10000,
        years=2  # Full backtest
    )

    # Calculate metrics
    avg_reward = np.mean(rewards)
    avg_final_value = np.mean(values)
    success_rate = np.mean(np.array(rewards) > 0) * 100

    # Extract key backtest metrics
    backtest_metrics = backtest_results["metrics"]

    # Store results
    fine_tuning_result = {
        "model_name": model_name,
        "model_path": model_path,
        "original_model": best_model["model_name"],
        "parameters": best_params,
        "evaluation": {
            "avg_reward": float(avg_reward),
            "avg_final_value": float(avg_final_value),
            "success_rate": float(success_rate)
        },
        "backtest": {
            "total_return": float(backtest_metrics["total_return"]),
            "annualized_return": float(backtest_metrics["annualized_return"]),
            "volatility": float(backtest_metrics["volatility"]),
            "sharpe_ratio": float(backtest_metrics["sharpe_ratio"]),
            "max_drawdown": float(backtest_metrics["max_drawdown"]),
            "win_rate": float(backtest_metrics["win_rate"])
        },
        "portfolio_characteristics": {
            "avg_cash_allocation": float(np.mean(backtest_results["cash_allocations"]) * 100),
            "avg_holdings": float(np.mean(backtest_results["holdings_diversification"])),
            "avg_turnover": float(np.mean(backtest_results["daily_turnover"]) * 100)
        }
    }

    # Save results
    results_path = os.path.join(os.path.dirname(base_results_path), f"fine_tuning_results_{timestamp}.json")
    with open(results_path, "w") as f:
        json.dump(fine_tuning_result, f, indent=2)

    print(f"\nFine-tuning results saved to: {results_path}")
    print("\nFine-tuned model performance:")
    print(f"  Avg reward: {avg_reward:.4f}")
    print(f"  Avg final value: ${avg_final_value:.2f}")
    print(f"  Success rate: {success_rate:.2f}%")
    print(f"  Sharpe ratio: {backtest_metrics['sharpe_ratio']:.2f}")
    print(f"  Annualized return: {backtest_metrics['annualized_return']:.2f}%")
    print(f"  Max drawdown: {backtest_metrics['max_drawdown']:.2f}%")

    return fine_tuning_result


# Modified versions of evaluate_model and backtest_portfolio for tuning
def evaluate_model(model, data_path, max_stocks=100, lookback=30, episodes=10, verbose=True):
    """Evaluate the model on a validation dataset with option to silence output"""
    if verbose:
        print(f"Evaluating model performance...")

    eval_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=5  # Use more recent data for evaluation
    )

    eval_env = PortfolioTradingEnv(eval_iter)

    total_rewards = []
    portfolio_values = []

    for ep in range(episodes):
        obs, _ = eval_env.reset()
        done = False
        episode_reward = 0
        initial_value = eval_env.initial_cash
        step_count = 0

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _, info = eval_env.step(action)
            episode_reward += reward
            step_count += 1

            if done:
                final_value = info["portfolio_value"]
                portfolio_values.append(final_value)

                # Calculate annualized return
                if step_count > 0:
                    days = step_count
                    annualized_return = ((final_value / initial_value) ** (252 / days) - 1) * 100
                else:
                    annualized_return = 0

                if verbose:
                    print(f"Episode {ep+1}: Return = {episode_reward:.4f}, "
                          f"Steps = {step_count}, "
                          f"Final Value = ${final_value:.2f}, "
                          f"Annualized Return = {annualized_return:.2f}%")

        total_rewards.append(episode_reward)

    # Calculate summary statistics
    avg_reward = np.mean(total_rewards)
    avg_final_value = np.mean(portfolio_values)
    success_rate = np.sum(np.array(total_rewards) > 0) / len(total_rewards) * 100

    if verbose:
        print(f"\nEvaluation Results (over {episodes} episodes):")
        print(f"Average Total Reward: {avg_reward:.4f}")
        print(f"Average Final Portfolio Value: ${avg_final_value:.2f}")
        print(f"Success Rate (positive return): {success_rate:.2f}%")

    return total_rewards, portfolio_values


def backtest_portfolio(model, data_path, max_stocks=100, lookback=30, initial_cash=10000, years=2, verbose=True):
    """Run a comprehensive backtest of the portfolio strategy with option to silence output"""
    if verbose:
        print(f"Running {years}-year backtest simulation...")

    # Initialize the environment with the backtest data
    backtest_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=years
    )

    backtest_env = PortfolioTradingEnv(backtest_iter, initial_cash=initial_cash)

    # Run backtest
    obs, _ = backtest_env.reset()
    done = False

    portfolio_values = [initial_cash]
    returns = []
    cash_allocations = []
    transaction_costs = []
    holdings_diversification = []
    daily_turnover = []

    step = 0
    while not done:
        action, _ = model.predict(obs, deterministic=True)

        # Track portfolio composition
        cash_allocations.append(action[0])

        # Number of stocks with allocation > 1%
        significant_holdings = sum(1 for a in action[1:] if a > 0.01)
        holdings_diversification.append(significant_holdings)

        # Execute step
        obs, reward, done, _, info = backtest_env.step(action)

        # Track metrics
        portfolio_values.append(info["portfolio_value"])
        if step > 0:
            returns.append((portfolio_values[-1] / portfolio_values[-2]) - 1)
        transaction_costs.append(info["transaction_costs"])

        # Calculate turnover (sum of absolute changes in allocation)
        if step == 0:
            daily_turnover.append(0)
        else:
            turnover = info["transaction_costs"] / (portfolio_values[-2] * backtest_env.transaction_cost)
            daily_turnover.append(turnover)

        step += 1

        # Print progress
        if verbose and step % 50 == 0:
            print(f"Backtest step {step}, Portfolio value: ${portfolio_values[-1]:.2f}")

    # Calculate performance metrics
    metrics = calculate_portfolio_metrics(portfolio_values)

    backtest_results = {
        "portfolio_values": portfolio_values,
        "returns": returns,
        "cash_allocations": cash_allocations,
        "transaction_costs": transaction_costs,
        "holdings_diversification": holdings_diversification,
        "daily_turnover": daily_turnover,
        "metrics": metrics,
        "steps": step,
        "final_value": portfolio_values[-1]
    }

    return backtest_results


# Example usage:
if __name__ == "__main__":
    # Set up parameters for tuning
    DATA_PATH = "/content/historical_data.csv"
    TUNING_DIR = "tuning_models"

    # First round of hyperparameter tuning with reduced parameter space
    tuning_results = hyperparameter_tuning(
        DATA_PATH,
        base_model_path=TUNING_DIR,
        max_stocks_options=[50, 100],
        lookback_options=[15, 30],
        learning_rate_options=[1e-4, 3e-4],
        gamma_options=[0.99],
        ent_coef_options=[0.01],
        timesteps=10000,  # Reduced for initial tuning
        eval_episodes=3
    )

    # Get the path to the tuning results
    results_files = [f for f in os.listdir(os.path.join(TUNING_DIR, "results")) if f.startswith("tuning_results")]
    latest_results = sorted(results_files)[-1]
    results_path = os.path.join(TUNING_DIR, "results", latest_results)

    # Fine-tune the best model by Sharpe ratio
    fine_tuned_results = fine_tune_best_model(
        DATA_PATH,
        results_path,
        best_model_type="best_by_sharpe",
        timesteps=10000,  # Full training for the best model
        eval_episodes=10
    )

Tuning with 8 parameter combinations


Hyperparameter tuning:   0%|          | 0/8 [00:00<?, ?it/s]


Training model 1/8: model_1_ms50_lb15_lr0.0001_g0.99_e0.01
Loading data from /content/historical_data.csv...
Adding technical indicators...



Processing tickers: 100%|██████████| 50/50 [00:01<00:00, 48.62it/s]


Technical indicators added successfully
Scaling features...


...

Technical indicators added successfully
Scaling features...


Scaling features: 100%|██████████| 50/50 [00:00<00:00, 86.92it/s]


Preparing training data...


Processing dates: 100%|██████████| 502/502 [00:08<00:00, 60.03it/s]


Loaded 50 stocks with 502 trading days
Date range: 2022-12-30 00:00:00 to 2024-12-30 00:00:00
Backtest step 50, Portfolio value: $9424.59
Step 100: Reward = 0.010592, Portfolio Value: 8925.79, Transaction Costs: 4.72
Backtest step 100, Portfolio value: $8925.79
Backtest step 150, Portfolio value: $9451.61
Step 200: Reward = 0.003841, Portfolio Value: 9441.12, Transaction Costs: 2.89
Backtest step 200, Portfolio value: $9441.12
Backtest step 250, Portfolio value: $9799.83
Step 300: Reward = -0.002654, Portfolio Value: 11341.90, Transaction Costs: 2.77
Backtest step 300, Portfolio value: $11341.90
Backtest step 350, Portfolio value: $10863.57
Step 400: Reward = 0.005920, Portfolio Value: 11174.38, Transaction Costs: 3.94
Backtest step 400, Portfolio value: $11174.38
Backtest step 450, Portfolio value: $12322.91
Step 472: Reward = -0.000391, Portfolio Value: 11626.37, Transaction Costs: 2.27

Fine-tuning results saved to: tuning_models/results/fine_tuning_results_20250319_173939.json

Fin